## RP Code

In [5]:
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# -----------------------------
# Config
# -----------------------------
INPUT_PATH = "/home/prathamrao/Visualtization/Input/MaxQuant-analysis_all RP.xlsx"
PSEUDOCOUNT = 1.0
COLOR_RANGE = 10.0                  # symmetric clamp for colors, e.g. [-3, +3]
OUTPUT_HTML = "/home/prathamrao/Visualtization/Output_2/RP_Total_Proteomics.html"
PROTEIN_COL = "Protein names"
GENE_COL = "Gene names"

# Expected intensity columns like: "CNTRL 6h", "EBV 12h", "CD40 24h", "EAG 6h"
CONDITIONS = ["CNTRL", "EBV", "CD40", "EAG"]
TIMES = ["6h", "12h", "24h"]

# Pairwise contrasts (A vs B means log2((A+eps)/(B+eps)))
CONTRASTS = [
    ("EBV", "CNTRL"),
    ("EBV", "CD40"),
    ("EBV", "EAG"),
    ("EAG", "CNTRL"),
    ("CD40", "CNTRL"),
    ("EAG", "CD40")
]

# -----------------------------
# Helpers
# -----------------------------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Trim, collapse whitespace, and unify '24 h' -> '24h' in column names."""
    cols = [str(c).strip().replace("\xa0", " ") for c in df.columns]
    cols = [re.sub(r"\s+", " ", c) for c in cols]
    cols = [re.sub(r"(\d+)\s*h", r"\1h", c, flags=re.IGNORECASE) for c in cols]
    df = df.copy()
    df.columns = cols
    return df

def find_intensity_columns(df: pd.DataFrame):
    """
    Returns mapping: (condition, time) -> column_name
    Matches columns like 'CNTRL 6h' (after normalization).
    """
    mapping = {}
    for col in df.columns:
        m = re.match(r"^\s*(CNTRL|EBV|CD40|EAG)\s+(\d+h)\s*$", str(col), flags=re.IGNORECASE)
        if m:
            cond = m.group(1).upper()
            time = m.group(2).lower()
            mapping[(cond, time)] = col
    missing = [(c, t) for c in CONDITIONS for t in TIMES if (c, t.lower()) not in mapping]
    if missing:
        raise ValueError(f"Missing intensity columns for: {missing}\nFound keys: {sorted(mapping.keys())}")
    return mapping

def clean_labels(s: pd.Series) -> pd.Index:
    """Ensure clean, non-empty labels; fallback to 'UNKNOWN'."""
    g = s.astype(str).replace({"nan": "", "None": ""}).str.strip()
    g = g.where(g != "", None)
    g = pd.Series([x if x not in (None, "") else "UNKNOWN" for x in g])
    return pd.Index(g, name="Gene")

def compute_log2fc_matrix(df: pd.DataFrame, intensity_map: dict):
    """
    Builds a wide matrix with columns for each (contrast, time).
    Returns:
      - mat: DataFrame indexed by gene name, columns = heatmap columns (strings)
    """
    # Row labels: prefer gene names; fallback to protein names if all unknown
    row_label = clean_labels(df[GENE_COL])
    if (row_label == "UNKNOWN").all() and PROTEIN_COL in df.columns:
        row_label = clean_labels(df[PROTEIN_COL])

    out_data = {}
    columns = []
    for a, b in CONTRASTS:
        contrast_key = f"{a} vs {b}"
        for t in TIMES:
            a_col = intensity_map[(a, t)]
            b_col = intensity_map[(b, t)]
            a_vals = pd.to_numeric(df[a_col], errors="coerce").fillna(0.0).to_numpy()
            b_vals = pd.to_numeric(df[b_col], errors="coerce").fillna(0.0).to_numpy()
            log2fc = np.log2((a_vals + PSEUDOCOUNT) / (b_vals + PSEUDOCOUNT))
            col_name = f"{contrast_key} | {t}"
            out_data[col_name] = log2fc
            columns.append(col_name)

    mat = pd.DataFrame(out_data, index=row_label)
    # Collapse duplicate gene labels by median so y-axis shows one row per gene
    mat = mat.groupby(level=0).median()
    # Replace inf with NaN and drop all-NaN rows
    mat = mat.replace([np.inf, -np.inf], np.nan).dropna(how="all")
    # Keep columns in deterministic order
    mat = mat.loc[:, columns]
    return mat

# -----------------------------
# Main
# -----------------------------
def main():
    df = pd.read_excel(INPUT_PATH, sheet_name=0)
    df = normalize_columns(df)

    for required in (PROTEIN_COL, GENE_COL):
        if required not in df.columns:
            raise ValueError(f"Missing required column: '{required}'")
    
    column_names = ['CNTRL 6h', 'CNTRL 12h', 'CNTRL 24h', 'EBV 6h','EBV 12h', 'EBV 24h', 'CD40 6h', 'CD40 12h', 'CD40 24h', 'EAG 6h','EAG 12h', 'EAG 24h']
    for n in column_names:
        df[n] = df[n] * 1000
        df.loc[df[n]==0, n] = 0.001

    intensity_map = find_intensity_columns(df)
    mat = compute_log2fc_matrix(df, intensity_map)

    # Initial (base) order
    z0 = mat.to_numpy()
    x0 = mat.columns.tolist()
    y0 = [str(y) for y in mat.index.tolist()]  # force strings to avoid date inference

    # Build per-column row-sorted views (ascending and descending)
    sorted_views = {}
    for j, col in enumerate(x0):
        vals = mat[col].to_numpy()
        asc_idx = np.argsort(np.nan_to_num(vals, nan=np.inf))       # NaN at bottom
        desc_idx = np.argsort(np.nan_to_num(-vals, nan=np.inf))     # NaN at bottom for desc
        sorted_views[(col, "asc")] = (mat.values[asc_idx, :], [y0[i] for i in asc_idx])
        sorted_views[(col, "desc")] = (mat.values[desc_idx, :], [y0[i] for i in desc_idx])

    # Heatmap with colorbar on the left
    heatmap = go.Heatmap(
        z=z0,
        x=x0,
        y=y0,
        colorscale="RdBu",
        reversescale=True,
        zmid=0,
        zmin=-COLOR_RANGE,
        zmax=COLOR_RANGE,
        colorbar=dict(
            title=dict(text="log2FC", side="right"),
            x=-0.08,   # left of plotting area (paper coordinates)
            y=0.5,
            len=0.9,
            thickness=14
        ),
        hovertemplate="<b>%{y}</b><br>%{x}<br>log2FC=%{z:.3f}<extra></extra>",
    )
    fig = go.Figure(data=[heatmap])

    # Dropdown buttons: base + sort by each column (asc/desc)
    buttons = [
        dict(
            label="Base order",
            method="update",
            args=[
                {"z": [z0], "x": [x0], "y": [y0]},
                {"title": None},
            ],
        )
    ]
    for col in x0:
        for direction in ("asc", "desc"):
            z_i, y_i = sorted_views[(col, direction)]
            buttons.append(
                dict(
                    label=f"Sort by {col} ({'↑' if direction=='asc' else '↓'})",
                    method="update",
                    args=[
                        {"z": [z_i], "x": [x0], "y": [y_i]},
                        {"title": None},
                    ],
                )
            )

    # Layout: y-axis labels on right; place dropdown at top-right corner
    fig.update_yaxes(type="category", categoryorder="trace", side="right", autorange="reversed")
    fig.update_layout(
        title=None,
        xaxis=dict(tickangle=45),
        margin=dict(l=160, r=240, t=6, b=120),
        updatemenus=[
            dict(
                type="dropdown",
                x=1.0, y=1.0,                  # top-right corner
                xanchor="right", yanchor="top",
                pad=dict(t=0, b=0, l=0, r=0),
                buttons=buttons,
                showactive=True,
                direction="down",
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="rgba(0,0,0,0.2)",
                borderwidth=1
            )
        ],
        height=max(700, 16 * len(y0)),
    )

    # Build custom HTML with a filter panel (cutoff + direction + column select)
    fig_div = pio.to_html(fig, include_plotlyjs="cdn", full_html=False, div_id="plot")

    controls_html = """
<div id="filter-panel" style="display:flex; gap:12px; align-items:center; justify-content:flex-end; margin:6px 12px 12px 12px; font-family:Arial, sans-serif;">
  <label for="colSelect"><strong>Column:</strong></label>
  <select id="colSelect" style="max-width:420px; padding:4px 6px;"></select>
  <label for="direction"><strong>Direction:</strong></label>
  <select id="direction" style="padding:4px 6px;">
    <option value="gt">&gt; (greater than)</option>
    <option value="lt">&lt; (less than)</option>
  </select>
  <label for="cutoff"><strong>Cutoff:</strong></label>
  <input id="cutoff" type="number" step="0.01" value="1.0" style="width:100px; padding:4px 6px;" />
  <button id="applyFilter" style="padding:6px 10px; cursor:pointer;">Apply</button>
  <button id="downloadCsv" style="padding:6px 10px; cursor:pointer;">Download CSV</button>
</div>
<div id="results" style="margin:0 12px 16px 12px; font-family:Arial, sans-serif;">
  <div id="summary" style="margin-bottom:6px; color:#333;"></div>
  <div id="list" style="max-height:320px; overflow:auto; border:1px solid #ddd; padding:8px; border-radius:6px; background:#fafafa; user-select:text;"></div>
  <div id="copyHint" style="margin-top:6px; font-size:12px; color:#555;">Tip: click any column header to copy that column (e.g., click “Gene” to copy gene names).</div>
</div>
"""

    script = """
<script>
document.addEventListener("DOMContentLoaded", function() {
  const gd = document.getElementById("plot");
  const colSelect = document.getElementById("colSelect");
  const directionSel = document.getElementById("direction");
  const cutoffInput = document.getElementById("cutoff");
  const applyBtn = document.getElementById("applyFilter");
  const downloadBtn = document.getElementById("downloadCsv");
  const summary = document.getElementById("summary");
  const listDiv = document.getElementById("list");

  function populateColumns() {
    const x = gd && gd.data && gd.data[0] ? gd.data[0].x : [];
    colSelect.innerHTML = "";
    x.forEach((label, idx) => {
      const opt = document.createElement("option");
      opt.value = idx;
      opt.textContent = label;
      colSelect.appendChild(opt);
    });
  }

  function formatNumber(v) {
    if (v === null || v === undefined || isNaN(v)) return "";
    return Number(v).toFixed(3);
  }

  function buildTable(rows) {
    const tableHtml = `
      <table id="genesTable" style="width:100%; border-collapse:collapse; font-family:monospace;">
        <thead>
          <tr>
            <th data-col="0" style="text-align:left; padding:6px; border-bottom:1px solid #ccc; position:sticky; top:0; background:#f7f7f7; cursor:pointer;">
              Gene <span style="font-size:11px; color:#666; margin-left:6px;">(click to copy)</span>
            </th>
            <th data-col="1" style="text-align:right; padding:6px; border-bottom:1px solid #ccc; position:sticky; top:0; background:#f7f7f7; cursor:pointer;">
              Value <span style="font-size:11px; color:#666; margin-left:6px;">(click to copy)</span>
            </th>
          </tr>
        </thead>
        <tbody>
          ${rows.map(r => `
            <tr>
              <td style="padding:6px; border-bottom:1px solid #eee;">${r.gene}</td>
              <td style="padding:6px; text-align:right; border-bottom:1px solid #eee;">${formatNumber(r.value)}</td>
            </tr>
          `).join("")}
        </tbody>
      </table>
    `;
    listDiv.innerHTML = tableHtml;

    const tableEl = document.getElementById("genesTable");
    attachColumnInteractions(tableEl);
  }

  async function copyColumnText(tableEl, colIdx) {
    const cells = Array.from(tableEl.querySelectorAll(`tbody tr td:nth-child(${colIdx + 1})`));
    const text = cells.map(td => td.textContent.trim()).join("\\n");
    if (!text) return;
    try {
      if (navigator.clipboard && navigator.clipboard.writeText) {
        await navigator.clipboard.writeText(text);
      } else {
        const ta = document.createElement("textarea");
        ta.value = text;
        ta.style.position = "fixed";
        ta.style.opacity = "0";
        document.body.appendChild(ta);
        ta.select();
        document.execCommand("copy");
        document.body.removeChild(ta);
      }
      flashHeader(tableEl, colIdx, "Copied!");
    } catch (e) {
      console.warn("Copy failed:", e);
      flashHeader(tableEl, colIdx, "Copy failed");
    }
  }

  function flashHeader(tableEl, colIdx, msg) {
    const th = tableEl.querySelector(`thead th:nth-child(${colIdx + 1})`);
    if (!th) return;
    const old = th.innerHTML;
    th.innerHTML = th.innerHTML.replace(/<span[^>]*>.*?<\\/span>/, "") + ` <span style="font-size:11px; color:#0a0; margin-left:6px;">${msg}</span>`;
    setTimeout(() => { th.innerHTML = old; }, 1200);
  }

  function attachColumnInteractions(tableEl) {
    const headers = tableEl.querySelectorAll("thead th");
    headers.forEach((th, idx) => {
      // Hover highlight
      th.addEventListener("mouseenter", () => highlightColumn(tableEl, idx, true));
      th.addEventListener("mouseleave", () => highlightColumn(tableEl, idx, false));
      // Click-to-copy
      th.addEventListener("click", () => copyColumnText(tableEl, idx));
    });
  }

  function highlightColumn(tableEl, colIdx, on) {
    const header = tableEl.querySelector(`thead th:nth-child(${colIdx + 1})`);
    const cells = tableEl.querySelectorAll(`tbody tr td:nth-child(${colIdx + 1})`);
    if (header) header.style.background = on ? "#e9f5ff" : "#f7f7f7";
    cells.forEach(td => td.style.background = on ? "#f2faff" : "");
  }

  function applyFilter() {
    if (!gd || !gd.data || !gd.data[0]) return;
    const z = gd.data[0].z || [];
    const y = gd.data[0].y || [];
    const colIdx = parseInt(colSelect.value, 10);
    const direction = directionSel.value; // 'gt' or 'lt'
    const cutoff = parseFloat(cutoffInput.value);
    if (isNaN(colIdx) || isNaN(cutoff)) return;

    let rows = [];
    for (let i = 0; i < z.length; i++) {
      const v = z[i][colIdx];
      if (v == null || isNaN(v)) continue;
      if ((direction === "gt" && v > cutoff) || (direction === "lt" && v < cutoff)) {
        rows.push({ gene: String(y[i]), value: v });
      }
    }

    rows.sort((a, b) => direction === "gt" ? (b.value - a.value) : (a.value - b.value));

    // Replace your current summary.textContent line with this block inside applyFilter:

    const snippet = `${gd.data[0].x[colIdx]}, ${direction === "gt" ? ">" : "<"} ${cutoff}, NVS`;
    summary.innerHTML = `Matches: ${rows.length} genes (column: ${snippet}) `;

    const copyBtn = document.createElement("button");
    copyBtn.textContent = "Copy";
    copyBtn.title = "Copy column, direction, cutoff, NVS";
    copyBtn.style.marginLeft = "6px";
    copyBtn.style.padding = "2px 6px";
    copyBtn.style.fontSize = "12px";
    copyBtn.style.cursor = "pointer";

    copyBtn.addEventListener("click", async () => {
      try {
        if (navigator.clipboard && navigator.clipboard.writeText) {
          await navigator.clipboard.writeText(snippet);
        } else {
          const ta = document.createElement("textarea");
          ta.value = snippet;
          ta.style.position = "fixed";
          ta.style.opacity = "0";
          document.body.appendChild(ta);
          ta.select();
          document.execCommand("copy");
          document.body.removeChild(ta);
        }
        const old = copyBtn.textContent;
        copyBtn.textContent = "Copied!";
        setTimeout(() => (copyBtn.textContent = old), 900);
      } catch {
        const old = copyBtn.textContent;
        copyBtn.textContent = "Copy failed";
        setTimeout(() => (copyBtn.textContent = old), 1200);
      }
    });

    summary.appendChild(copyBtn);

    if (rows.length === 0) {
      listDiv.innerHTML = "<em>No genes found for this cutoff and direction.</em>";
    } else {
      buildTable(rows);
    }

    // Prepare CSV for download
    const csvHeader = "Gene,Value\\n";
    const csvBody = rows.map(r => `${r.gene.replace(/"/g,'""')},${r.value}`).join("\\n");
    const csv = csvHeader + csvBody;
    const blob = new Blob([csv], { type: "text/csv" });
    const url = URL.createObjectURL(blob);
    downloadBtn.dataset.url = url;
    downloadBtn.dataset.filename = `genes_${gd.data[0].x[colIdx].replace(/\\s+/g,'_')}_${direction}_${cutoff}.csv`;
  }

  downloadBtn.addEventListener("click", function() {
    const url = this.dataset.url;
    const name = this.dataset.filename || "genes.csv";
    if (!url) return;
    const a = document.createElement("a");
    a.href = url;
    a.download = name;
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
  });

  applyBtn.addEventListener("click", applyFilter);

  function init() {
    populateColumns();
    applyFilter();
  }
  if (gd && gd.data) {
    init();
  } else {
    setTimeout(init, 100);
  }
});
</script>
"""

    html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>log2FC Heatmap with Filtering</title>
<style>
  body {{ margin: 8px; }}
</style>
</head>
<body>
{controls_html}
{fig_div}
{script}
</body>
</html>
"""

    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Wrote: {OUTPUT_HTML}")

if __name__ == "__main__":
    main()

Wrote: /home/prathamrao/Visualtization/Output_2/RP_Total_Proteomics.html


## NVSCode

In [4]:
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# -----------------------------
# Config
# -----------------------------
INPUT_PATH = "/home/prathamrao/Visualtization/Input/MaxQuant-analysis_all NVS.xlsx"
PSEUDOCOUNT = 1.0
COLOR_RANGE = 10.0                  # symmetric clamp for colors, e.g. [-3, +3]
OUTPUT_HTML = "/home/prathamrao/Visualtization/Output_2/NVS_Total_Proteomics.html"
PROTEIN_COL = "Protein names"
GENE_COL = "Gene names"

# Expected intensity columns like: "CNTRL 6h", "EBV 12h", "CD40 24h", "EAG 6h"
CONDITIONS = ["CNTRL", "EBV", "CD40", "EAG"]
TIMES = ["6h", "12h", "24h"]

# Pairwise contrasts (A vs B means log2((A+eps)/(B+eps)))
CONTRASTS = [
    ("EBV", "CNTRL"),
    ("EBV", "CD40"),
    ("EBV", "EAG"),
    ("EAG", "CNTRL"),
    ("CD40", "CNTRL"),
    ("EAG", "CD40")
]

# -----------------------------
# Helpers
# -----------------------------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Trim, collapse whitespace, and unify '24 h' -> '24h' in column names."""
    cols = [str(c).strip().replace("\xa0", " ") for c in df.columns]
    cols = [re.sub(r"\s+", " ", c) for c in cols]
    cols = [re.sub(r"(\d+)\s*h", r"\1h", c, flags=re.IGNORECASE) for c in cols]
    df = df.copy()
    df.columns = cols
    return df

def find_intensity_columns(df: pd.DataFrame):
    """
    Returns mapping: (condition, time) -> column_name
    Matches columns like 'CNTRL 6h' (after normalization).
    """
    mapping = {}
    for col in df.columns:
        m = re.match(r"^\s*(CNTRL|EBV|CD40|EAG)\s+(\d+h)\s*$", str(col), flags=re.IGNORECASE)
        if m:
            cond = m.group(1).upper()
            time = m.group(2).lower()
            mapping[(cond, time)] = col
    missing = [(c, t) for c in CONDITIONS for t in TIMES if (c, t.lower()) not in mapping]
    if missing:
        raise ValueError(f"Missing intensity columns for: {missing}\nFound keys: {sorted(mapping.keys())}")
    return mapping

def clean_labels(s: pd.Series) -> pd.Index:
    """Ensure clean, non-empty labels; fallback to 'UNKNOWN'."""
    g = s.astype(str).replace({"nan": "", "None": ""}).str.strip()
    g = g.where(g != "", None)
    g = pd.Series([x if x not in (None, "") else "UNKNOWN" for x in g])
    return pd.Index(g, name="Gene")

def compute_log2fc_matrix(df: pd.DataFrame, intensity_map: dict):
    """
    Builds a wide matrix with columns for each (contrast, time).
    Returns:
      - mat: DataFrame indexed by gene name, columns = heatmap columns (strings)
    """
    # Row labels: prefer gene names; fallback to protein names if all unknown
    row_label = clean_labels(df[GENE_COL])
    if (row_label == "UNKNOWN").all() and PROTEIN_COL in df.columns:
        row_label = clean_labels(df[PROTEIN_COL])

    out_data = {}
    columns = []
    for a, b in CONTRASTS:
        contrast_key = f"{a} vs {b}"
        for t in TIMES:
            a_col = intensity_map[(a, t)]
            b_col = intensity_map[(b, t)]
            a_vals = pd.to_numeric(df[a_col], errors="coerce").fillna(0.0).to_numpy()
            b_vals = pd.to_numeric(df[b_col], errors="coerce").fillna(0.0).to_numpy()
            log2fc = np.log2((a_vals + PSEUDOCOUNT) / (b_vals + PSEUDOCOUNT))
            col_name = f"{contrast_key} | {t}"
            out_data[col_name] = log2fc
            columns.append(col_name)

    mat = pd.DataFrame(out_data, index=row_label)
    # Collapse duplicate gene labels by median so y-axis shows one row per gene
    mat = mat.groupby(level=0).median()
    # Replace inf with NaN and drop all-NaN rows
    mat = mat.replace([np.inf, -np.inf], np.nan).dropna(how="all")
    # Keep columns in deterministic order
    mat = mat.loc[:, columns]
    return mat

# -----------------------------
# Main
# -----------------------------
def main():
    df = pd.read_excel(INPUT_PATH, sheet_name=0)
    df = normalize_columns(df)

    for required in (PROTEIN_COL, GENE_COL):
        if required not in df.columns:
            raise ValueError(f"Missing required column: '{required}'")
    
    column_names = ['CNTRL 6h', 'CNTRL 12h', 'CNTRL 24h', 'EBV 6h','EBV 12h', 'EBV 24h', 'CD40 6h', 'CD40 12h', 'CD40 24h', 'EAG 6h','EAG 12h', 'EAG 24h']
    for n in column_names:
        df[n] = df[n] * 1000
        df.loc[df[n]==0, n] = 0.001

    intensity_map = find_intensity_columns(df)
    mat = compute_log2fc_matrix(df, intensity_map)

    # Initial (base) order
    z0 = mat.to_numpy()
    x0 = mat.columns.tolist()
    y0 = [str(y) for y in mat.index.tolist()]  # force strings to avoid date inference

    # Build per-column row-sorted views (ascending and descending)
    sorted_views = {}
    for j, col in enumerate(x0):
        vals = mat[col].to_numpy()
        asc_idx = np.argsort(np.nan_to_num(vals, nan=np.inf))       # NaN at bottom
        desc_idx = np.argsort(np.nan_to_num(-vals, nan=np.inf))     # NaN at bottom for desc
        sorted_views[(col, "asc")] = (mat.values[asc_idx, :], [y0[i] for i in asc_idx])
        sorted_views[(col, "desc")] = (mat.values[desc_idx, :], [y0[i] for i in desc_idx])

    # Heatmap with colorbar on the left
    heatmap = go.Heatmap(
        z=z0,
        x=x0,
        y=y0,
        colorscale="RdBu",
        reversescale=True,
        zmid=0,
        zmin=-COLOR_RANGE,
        zmax=COLOR_RANGE,
        colorbar=dict(
            title=dict(text="log2FC", side="right"),
            x=-0.08,   # left of plotting area (paper coordinates)
            y=0.5,
            len=0.9,
            thickness=14
        ),
        hovertemplate="<b>%{y}</b><br>%{x}<br>log2FC=%{z:.3f}<extra></extra>",
    )
    fig = go.Figure(data=[heatmap])

    # Dropdown buttons: base + sort by each column (asc/desc)
    buttons = [
        dict(
            label="Base order",
            method="update",
            args=[
                {"z": [z0], "x": [x0], "y": [y0]},
                {"title": None},
            ],
        )
    ]
    for col in x0:
        for direction in ("asc", "desc"):
            z_i, y_i = sorted_views[(col, direction)]
            buttons.append(
                dict(
                    label=f"Sort by {col} ({'↑' if direction=='asc' else '↓'})",
                    method="update",
                    args=[
                        {"z": [z_i], "x": [x0], "y": [y_i]},
                        {"title": None},
                    ],
                )
            )

    # Layout: y-axis labels on right; place dropdown at top-right corner
    fig.update_yaxes(type="category", categoryorder="trace", side="right", autorange="reversed")
    fig.update_layout(
        title=None,
        xaxis=dict(tickangle=45),
        margin=dict(l=160, r=240, t=6, b=120),
        updatemenus=[
            dict(
                type="dropdown",
                x=1.0, y=1.0,                  # top-right corner
                xanchor="right", yanchor="top",
                pad=dict(t=0, b=0, l=0, r=0),
                buttons=buttons,
                showactive=True,
                direction="down",
                bgcolor="rgba(255,255,255,0.8)",
                bordercolor="rgba(0,0,0,0.2)",
                borderwidth=1
            )
        ],
        height=max(700, 16 * len(y0)),
    )

    # Build custom HTML with a filter panel (cutoff + direction + column select)
    fig_div = pio.to_html(fig, include_plotlyjs="cdn", full_html=False, div_id="plot")

    controls_html = """
<div id="filter-panel" style="display:flex; gap:12px; align-items:center; justify-content:flex-end; margin:6px 12px 12px 12px; font-family:Arial, sans-serif;">
  <label for="colSelect"><strong>Column:</strong></label>
  <select id="colSelect" style="max-width:420px; padding:4px 6px;"></select>
  <label for="direction"><strong>Direction:</strong></label>
  <select id="direction" style="padding:4px 6px;">
    <option value="gt">&gt; (greater than)</option>
    <option value="lt">&lt; (less than)</option>
  </select>
  <label for="cutoff"><strong>Cutoff:</strong></label>
  <input id="cutoff" type="number" step="0.01" value="1.0" style="width:100px; padding:4px 6px;" />
  <button id="applyFilter" style="padding:6px 10px; cursor:pointer;">Apply</button>
  <button id="downloadCsv" style="padding:6px 10px; cursor:pointer;">Download CSV</button>
</div>
<div id="results" style="margin:0 12px 16px 12px; font-family:Arial, sans-serif;">
  <div id="summary" style="margin-bottom:6px; color:#333;"></div>
  <div id="list" style="max-height:320px; overflow:auto; border:1px solid #ddd; padding:8px; border-radius:6px; background:#fafafa; user-select:text;"></div>
  <div id="copyHint" style="margin-top:6px; font-size:12px; color:#555;">Tip: click any column header to copy that column (e.g., click “Gene” to copy gene names).</div>
</div>
"""

    script = """
<script>
document.addEventListener("DOMContentLoaded", function() {
  const gd = document.getElementById("plot");
  const colSelect = document.getElementById("colSelect");
  const directionSel = document.getElementById("direction");
  const cutoffInput = document.getElementById("cutoff");
  const applyBtn = document.getElementById("applyFilter");
  const downloadBtn = document.getElementById("downloadCsv");
  const summary = document.getElementById("summary");
  const listDiv = document.getElementById("list");

  function populateColumns() {
    const x = gd && gd.data && gd.data[0] ? gd.data[0].x : [];
    colSelect.innerHTML = "";
    x.forEach((label, idx) => {
      const opt = document.createElement("option");
      opt.value = idx;
      opt.textContent = label;
      colSelect.appendChild(opt);
    });
  }

  function formatNumber(v) {
    if (v === null || v === undefined || isNaN(v)) return "";
    return Number(v).toFixed(3);
  }

  function buildTable(rows) {
    const tableHtml = `
      <table id="genesTable" style="width:100%; border-collapse:collapse; font-family:monospace;">
        <thead>
          <tr>
            <th data-col="0" style="text-align:left; padding:6px; border-bottom:1px solid #ccc; position:sticky; top:0; background:#f7f7f7; cursor:pointer;">
              Gene <span style="font-size:11px; color:#666; margin-left:6px;">(click to copy)</span>
            </th>
            <th data-col="1" style="text-align:right; padding:6px; border-bottom:1px solid #ccc; position:sticky; top:0; background:#f7f7f7; cursor:pointer;">
              Value <span style="font-size:11px; color:#666; margin-left:6px;">(click to copy)</span>
            </th>
          </tr>
        </thead>
        <tbody>
          ${rows.map(r => `
            <tr>
              <td style="padding:6px; border-bottom:1px solid #eee;">${r.gene}</td>
              <td style="padding:6px; text-align:right; border-bottom:1px solid #eee;">${formatNumber(r.value)}</td>
            </tr>
          `).join("")}
        </tbody>
      </table>
    `;
    listDiv.innerHTML = tableHtml;

    const tableEl = document.getElementById("genesTable");
    attachColumnInteractions(tableEl);
  }

  async function copyColumnText(tableEl, colIdx) {
    const cells = Array.from(tableEl.querySelectorAll(`tbody tr td:nth-child(${colIdx + 1})`));
    const text = cells.map(td => td.textContent.trim()).join("\\n");
    if (!text) return;
    try {
      if (navigator.clipboard && navigator.clipboard.writeText) {
        await navigator.clipboard.writeText(text);
      } else {
        const ta = document.createElement("textarea");
        ta.value = text;
        ta.style.position = "fixed";
        ta.style.opacity = "0";
        document.body.appendChild(ta);
        ta.select();
        document.execCommand("copy");
        document.body.removeChild(ta);
      }
      flashHeader(tableEl, colIdx, "Copied!");
    } catch (e) {
      console.warn("Copy failed:", e);
      flashHeader(tableEl, colIdx, "Copy failed");
    }
  }

  function flashHeader(tableEl, colIdx, msg) {
    const th = tableEl.querySelector(`thead th:nth-child(${colIdx + 1})`);
    if (!th) return;
    const old = th.innerHTML;
    th.innerHTML = th.innerHTML.replace(/<span[^>]*>.*?<\\/span>/, "") + ` <span style="font-size:11px; color:#0a0; margin-left:6px;">${msg}</span>`;
    setTimeout(() => { th.innerHTML = old; }, 1200);
  }

  function attachColumnInteractions(tableEl) {
    const headers = tableEl.querySelectorAll("thead th");
    headers.forEach((th, idx) => {
      // Hover highlight
      th.addEventListener("mouseenter", () => highlightColumn(tableEl, idx, true));
      th.addEventListener("mouseleave", () => highlightColumn(tableEl, idx, false));
      // Click-to-copy
      th.addEventListener("click", () => copyColumnText(tableEl, idx));
    });
  }

  function highlightColumn(tableEl, colIdx, on) {
    const header = tableEl.querySelector(`thead th:nth-child(${colIdx + 1})`);
    const cells = tableEl.querySelectorAll(`tbody tr td:nth-child(${colIdx + 1})`);
    if (header) header.style.background = on ? "#e9f5ff" : "#f7f7f7";
    cells.forEach(td => td.style.background = on ? "#f2faff" : "");
  }

  function applyFilter() {
    if (!gd || !gd.data || !gd.data[0]) return;
    const z = gd.data[0].z || [];
    const y = gd.data[0].y || [];
    const colIdx = parseInt(colSelect.value, 10);
    const direction = directionSel.value; // 'gt' or 'lt'
    const cutoff = parseFloat(cutoffInput.value);
    if (isNaN(colIdx) || isNaN(cutoff)) return;

    let rows = [];
    for (let i = 0; i < z.length; i++) {
      const v = z[i][colIdx];
      if (v == null || isNaN(v)) continue;
      if ((direction === "gt" && v > cutoff) || (direction === "lt" && v < cutoff)) {
        rows.push({ gene: String(y[i]), value: v });
      }
    }

    rows.sort((a, b) => direction === "gt" ? (b.value - a.value) : (a.value - b.value));

    // Replace your current summary.textContent line with this block inside applyFilter:

    const snippet = `${gd.data[0].x[colIdx]}, ${direction === "gt" ? ">" : "<"} ${cutoff}, NVS`;
    summary.innerHTML = `Matches: ${rows.length} genes (column: ${snippet}) `;

    const copyBtn = document.createElement("button");
    copyBtn.textContent = "Copy";
    copyBtn.title = "Copy column, direction, cutoff, NVS";
    copyBtn.style.marginLeft = "6px";
    copyBtn.style.padding = "2px 6px";
    copyBtn.style.fontSize = "12px";
    copyBtn.style.cursor = "pointer";

    copyBtn.addEventListener("click", async () => {
      try {
        if (navigator.clipboard && navigator.clipboard.writeText) {
          await navigator.clipboard.writeText(snippet);
        } else {
          const ta = document.createElement("textarea");
          ta.value = snippet;
          ta.style.position = "fixed";
          ta.style.opacity = "0";
          document.body.appendChild(ta);
          ta.select();
          document.execCommand("copy");
          document.body.removeChild(ta);
        }
        const old = copyBtn.textContent;
        copyBtn.textContent = "Copied!";
        setTimeout(() => (copyBtn.textContent = old), 900);
      } catch {
        const old = copyBtn.textContent;
        copyBtn.textContent = "Copy failed";
        setTimeout(() => (copyBtn.textContent = old), 1200);
      }
    });

    summary.appendChild(copyBtn);

    if (rows.length === 0) {
      listDiv.innerHTML = "<em>No genes found for this cutoff and direction.</em>";
    } else {
      buildTable(rows);
    }

    // Prepare CSV for download
    const csvHeader = "Gene,Value\\n";
    const csvBody = rows.map(r => `${r.gene.replace(/"/g,'""')},${r.value}`).join("\\n");
    const csv = csvHeader + csvBody;
    const blob = new Blob([csv], { type: "text/csv" });
    const url = URL.createObjectURL(blob);
    downloadBtn.dataset.url = url;
    downloadBtn.dataset.filename = `genes_${gd.data[0].x[colIdx].replace(/\\s+/g,'_')}_${direction}_${cutoff}.csv`;
  }

  downloadBtn.addEventListener("click", function() {
    const url = this.dataset.url;
    const name = this.dataset.filename || "genes.csv";
    if (!url) return;
    const a = document.createElement("a");
    a.href = url;
    a.download = name;
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
  });

  applyBtn.addEventListener("click", applyFilter);

  function init() {
    populateColumns();
    applyFilter();
  }
  if (gd && gd.data) {
    init();
  } else {
    setTimeout(init, 100);
  }
});
</script>
"""

    html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>log2FC Heatmap with Filtering</title>
<style>
  body {{ margin: 8px; }}
</style>
</head>
<body>
{controls_html}
{fig_div}
{script}
</body>
</html>
"""

    with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Wrote: {OUTPUT_HTML}")

if __name__ == "__main__":
    main()

Wrote: /home/prathamrao/Visualtization/Output_2/NVS_Total_Proteomics.html
